# 06. Institutional ATR Risk Management & Backtest Engine

### Upgrading from Toy Simulation to Quantitative Realism
Standard backtesting often uses zero-friction assumptions and unrealistically small sample sizes. This upgraded engine implements **Institutional Quant Desk Standards**:
- **Multi-Cycle Horizon**: Scaled candle limit per asset to ensure statistical validity.
- **Realistic Execution Friction**: Models exchange taker fees (0.075% per leg), bid-ask slippage (0.03%), and Stop-Loss market order gap slippage penalties.
- **Dynamic Stop Modes**: Evaluates **Static ATR**, **Break-Even Stop (+1R)**, and **ATR Trailing Stop**.
- **Portfolio Sizing & Equity Compounding**: Fixed fractional 1% account equity risk management starting at $10,000.
- **24/7 Crypto Annualization**: Proper 365-day annualization for Sharpe & Sortino ratios on resampled equity curves.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from backend.binance_client import binance_client
from backend.indicators import enrich_klines_dataframe
import time
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 220)
print('All modules loaded successfully.')

All modules loaded successfully.


## 1. Fetch Market Data for Simulation (1,000 Candles per Asset)
We pull 1,000 candles across major crypto assets (BTC, ETH, SOL, BNB, XRP, DOGE) on **15m and 1h timeframes**.

In [2]:
SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'XRPUSDT', 'DOGEUSDT']
TIMEFRAMES = ['15m', '1h']
LIMIT = 1000

datasets = {}
for tf in TIMEFRAMES:
    datasets[tf] = {}
    print(f'Fetching {tf} data (Limit: {LIMIT} candles)...')
    for sym in SYMBOLS:
        try:
            raw = binance_client.get_klines(sym, interval=tf, limit=LIMIT)
            df = enrich_klines_dataframe(raw)
            if not df.empty:
                datasets[tf][sym] = df
                print(f'  {sym} ({tf}): {len(df)} candles loaded ({df.index[0]} to {df.index[-1]})')
        except Exception as e:
            print(f'  {sym} ({tf}) Error: {e}')
        time.sleep(0.1)

print('\nData loading complete.')

Fetching 15m data (Limit: 1000 candles)...
  BTCUSDT (15m): 1000 candles loaded (0 to 999)
  ETHUSDT (15m): 1000 candles loaded (0 to 999)
  SOLUSDT (15m): 1000 candles loaded (0 to 999)
  BNBUSDT (15m): 1000 candles loaded (0 to 999)
  XRPUSDT (15m): 1000 candles loaded (0 to 999)
  DOGEUSDT (15m): 1000 candles loaded (0 to 999)
Fetching 1h data (Limit: 1000 candles)...
  BTCUSDT (1h): 1000 candles loaded (0 to 999)
  ETHUSDT (1h): 1000 candles loaded (0 to 999)
  SOLUSDT (1h): 1000 candles loaded (0 to 999)
  BNBUSDT (1h): 1000 candles loaded (0 to 999)
  XRPUSDT (1h): 1000 candles loaded (0 to 999)
  DOGEUSDT (1h): 1000 candles loaded (0 to 999)

Data loading complete.


## 2. Define Strategies to Backtest

In [3]:
def strat_supertrend_bullish(df):
    return (df['supertrend_dir'].shift(1) == -1) & (df['supertrend_dir'] == 1)

def strat_supertrend_bearish(df):
    return (df['supertrend_dir'].shift(1) == 1) & (df['supertrend_dir'] == -1)

def strat_stoch_rsi_oversold(df):
    return (df['stoch_k'].shift(1) <= df['stoch_d'].shift(1)) & (df['stoch_k'] > df['stoch_d']) & (df['stoch_k'] < 20)

def strat_stoch_rsi_overbought(df):
    return (df['stoch_k'].shift(1) >= df['stoch_d'].shift(1)) & (df['stoch_k'] < df['stoch_d']) & (df['stoch_k'] > 80)

def strat_bb_squeeze_breakout(df):
    squeeze_on = (df['bb_upper'] < df['kc_upper']) & (df['bb_lower'] > df['kc_lower'])
    return squeeze_on & (df['close'] > df['bb_upper'])

def strat_crypto_fib_stack(df):
    aligned = (df['ema_8'] > df['ema_21']) & (df['ema_21'] > df['ema_55'])
    prev = (df['ema_8'].shift(1) > df['ema_21'].shift(1)) & (df['ema_21'].shift(1) > df['ema_55'].shift(1))
    return aligned & (~prev)

def strat_vwma_cross(df):
    return (df['vwma_8'].shift(1) <= df['vwma_21'].shift(1)) & (df['vwma_8'] > df['vwma_21'])

def strat_rsi_oversold(df):
    return (df['rsi'] < 30) & (df['rsi'].shift(1) >= 30)

def strat_macd_bull_cross(df):
    return (df['macd_hist'].shift(1) <= 0) & (df['macd_hist'] > 0)

STRATEGIES = {
    'Supertrend Bullish (10,3)':   {'fn': strat_supertrend_bullish, 'direction': 'LONG'},
    'Supertrend Bearish (10,3)':   {'fn': strat_supertrend_bearish, 'direction': 'SHORT'},
    'StochRSI Oversold':           {'fn': strat_stoch_rsi_oversold, 'direction': 'LONG'},
    'StochRSI Overbought':         {'fn': strat_stoch_rsi_overbought, 'direction': 'SHORT'},
    'BB Squeeze Breakout':         {'fn': strat_bb_squeeze_breakout, 'direction': 'LONG'},
    'Crypto Fib Stack (8/21/55)':  {'fn': strat_crypto_fib_stack,   'direction': 'LONG'},
    'VWMA Cross (8/21 Volume)':    {'fn': strat_vwma_cross,         'direction': 'LONG'},
    'RSI Oversold Bounce':         {'fn': strat_rsi_oversold,        'direction': 'LONG'},
    'Bullish MACD Cross':          {'fn': strat_macd_bull_cross,     'direction': 'LONG'},
}

print(f'{len(STRATEGIES)} key strategies loaded for SL/TP evaluation.')

9 key strategies loaded for SL/TP evaluation.


## 3. Institutional Trade Simulator Engine (Friction + Dynamic Stops)

### Key Friction Parameters:
- **Taker Fee**: 0.075% per trade leg (0.15% roundtrip).
- **Bid-Ask Slippage**: 0.03% per trade leg (0.06% roundtrip).
- **SL Gap Penalty**: 0.1x ATR price decay past Stop-Loss level to model market order gap slippage.

### Dynamic Stop Modes Evaluated:
1. **Static**: Fixed SL and TP.
2. **Break-Even**: Move SL to entry price once trade reaches +1.0R profit.
3. **Trailing**: Trail SL by 1.5x ATR as market moves in favor.

In [4]:
def simulate_trade_institutional(df, entry_idx, direction, sl_mult=1.5, tp_mult=3.0, 
                                max_bars=40, fee_pct=0.00075, slippage_pct=0.0003, 
                                stop_mode='static'):
    """
    Simulates a single trade with realistic friction and dynamic stop modes.
    Returns net return %, bars held, and outcome ('TP', 'SL', 'TIMEOUT').
    """
    entry_row = df.iloc[entry_idx]
    raw_entry = entry_row['close']
    atr = entry_row['atr']
    if pd.isna(atr) or atr <= 0:
        atr = raw_entry * 0.01

    # Apply entry slippage
    if direction == 'LONG':
        entry_price = raw_entry * (1 + slippage_pct)
        initial_sl = entry_price - (sl_mult * atr)
        tp_price = entry_price + (tp_mult * atr)
    else:
        entry_price = raw_entry * (1 - slippage_pct)
        initial_sl = entry_price + (sl_mult * atr)
        tp_price = entry_price - (tp_mult * atr)

    current_sl = initial_sl
    risk_dist = abs(entry_price - initial_sl)
    be_triggered = False

    total_rows = len(df)
    for i in range(entry_idx + 1, min(entry_idx + 1 + max_bars, total_rows)):
        candle = df.iloc[i]
        high, low = candle['high'], candle['low']

        if direction == 'LONG':
            if stop_mode in ['breakeven', 'trailing'] and not be_triggered:
                if high >= entry_price + risk_dist:
                    current_sl = max(current_sl, entry_price)
                    be_triggered = True

            if stop_mode == 'trailing':
                current_sl = max(current_sl, high - (sl_mult * atr))

            hit_sl = low <= current_sl
            hit_tp = high >= tp_price

            if hit_sl and hit_tp:
                fill_price = current_sl - (0.1 * atr)
                gross_ret = (fill_price - entry_price) / entry_price
                net_ret = gross_ret - (2 * fee_pct) - (2 * slippage_pct)
                return {'outcome': 'SL', 'return_pct': net_ret * 100, 'bars_held': i - entry_idx}
            elif hit_tp:
                fill_price = tp_price
                gross_ret = (fill_price - entry_price) / entry_price
                net_ret = gross_ret - (2 * fee_pct) - (2 * slippage_pct)
                return {'outcome': 'TP', 'return_pct': net_ret * 100, 'bars_held': i - entry_idx}
            elif hit_sl:
                fill_price = current_sl - (0.1 * atr)
                gross_ret = (fill_price - entry_price) / entry_price
                net_ret = gross_ret - (2 * fee_pct) - (2 * slippage_pct)
                return {'outcome': 'SL', 'return_pct': net_ret * 100, 'bars_held': i - entry_idx}
        else:  # SHORT
            if stop_mode in ['breakeven', 'trailing'] and not be_triggered:
                if low <= entry_price - risk_dist:
                    current_sl = min(current_sl, entry_price)
                    be_triggered = True

            if stop_mode == 'trailing':
                current_sl = min(current_sl, low + (sl_mult * atr))

            hit_sl = high >= current_sl
            hit_tp = low <= tp_price

            if hit_sl and hit_tp:
                fill_price = current_sl + (0.1 * atr)
                gross_ret = (entry_price - fill_price) / entry_price
                net_ret = gross_ret - (2 * fee_pct) - (2 * slippage_pct)
                return {'outcome': 'SL', 'return_pct': net_ret * 100, 'bars_held': i - entry_idx}
            elif hit_tp:
                fill_price = tp_price
                gross_ret = (entry_price - fill_price) / entry_price
                net_ret = gross_ret - (2 * fee_pct) - (2 * slippage_pct)
                return {'outcome': 'TP', 'return_pct': net_ret * 100, 'bars_held': i - entry_idx}
            elif hit_sl:
                fill_price = current_sl + (0.1 * atr)
                gross_ret = (entry_price - fill_price) / entry_price
                net_ret = gross_ret - (2 * fee_pct) - (2 * slippage_pct)
                return {'outcome': 'SL', 'return_pct': net_ret * 100, 'bars_held': i - entry_idx}

    last_candle = df.iloc[min(entry_idx + max_bars, total_rows - 1)]
    exit_price = last_candle['close']
    if direction == 'LONG':
        gross_ret = (exit_price - entry_price) / entry_price
    else:
        gross_ret = (entry_price - exit_price) / entry_price
    net_ret = gross_ret - (2 * fee_pct) - (2 * slippage_pct)
    return {'outcome': 'TIMEOUT', 'return_pct': net_ret * 100, 'bars_held': max_bars}

print('Institutional Trade Simulator Engine ready.')

Institutional Trade Simulator Engine ready.


## 4. Run Risk Management Backtest across R:R Configurations & Stop Modes

In [5]:
RISK_MODES = ['static', 'breakeven', 'trailing']
RR_CONFIGS = [
    {'label': '1:1.5 R:R', 'sl_mult': 1.5, 'tp_mult': 2.25},
    {'label': '1:2.0 R:R', 'sl_mult': 1.5, 'tp_mult': 3.00},
    {'label': '1:3.0 R:R', 'sl_mult': 1.0, 'tp_mult': 3.00},
]

backtest_results = []

for tf in TIMEFRAMES:
    sym_dict = datasets.get(tf, {})
    for strat_name, strat_info in STRATEGIES.items():
        fn = strat_info['fn']
        direction = strat_info['direction']
        
        for rr in RR_CONFIGS:
            for mode in RISK_MODES:
                all_trades = []
                
                for sym, df in sym_dict.items():
                    signals = fn(df)
                    signal_indices = np.where(signals)[0]
                    
                    for idx in signal_indices:
                        if idx < len(df) - 5:
                            trade = simulate_trade_institutional(
                                df, idx, direction,
                                sl_mult=rr['sl_mult'],
                                tp_mult=rr['tp_mult'],
                                stop_mode=mode,
                                fee_pct=0.00075,
                                slippage_pct=0.0003
                            )
                            all_trades.append(trade)
                
                if len(all_trades) >= 5:
                    rets = pd.Series([t['return_pct'] for t in all_trades])
                    outcomes = [t['outcome'] for t in all_trades]
                    
                    tp_count = outcomes.count('TP')
                    sl_count = outcomes.count('SL')
                    timeout_count = outcomes.count('TIMEOUT')
                    
                    wins = rets[rets > 0]
                    losses = rets[rets <= 0]
                    win_rate = (len(wins) / len(rets)) * 100
                    
                    gross_profit = wins.sum() if len(wins) > 0 else 0
                    gross_loss = abs(losses.sum()) if len(losses) > 0 else 1e-10
                    profit_factor = gross_profit / gross_loss
                    
                    avg_win = wins.mean() if len(wins) > 0 else 0
                    avg_loss = abs(losses.mean()) if len(losses) > 0 else 0
                    expectancy = ((win_rate/100) * avg_win) - (((100-win_rate)/100) * avg_loss)
                    
                    # Portfolio Equity Curve simulation (1% risk per trade, initial $10k)
                    equity = [10000.0]
                    for r in rets:
                        trade_pnl = equity[-1] * 0.01 * (r / (rr['sl_mult'] * 1.0))
                        equity.append(equity[-1] + trade_pnl)
                    
                    eq_series = pd.Series(equity)
                    cum_return = ((eq_series.iloc[-1] - 10000.0) / 10000.0) * 100
                    dd = ((eq_series - eq_series.cummax()) / eq_series.cummax()) * 100
                    max_dd = dd.min()
                    
                    trade_std = rets.std() if len(rets) > 1 else 0
                    downside_std = rets[rets < 0].std() if len(rets[rets < 0]) > 1 else 1e-10
                    
                    sharpe = (rets.mean() / trade_std) * np.sqrt(365) if trade_std > 0 else 0
                    sortino = (rets.mean() / downside_std) * np.sqrt(365) if downside_std > 0 else 0
                    
                    backtest_results.append({
                        'Strategy': strat_name,
                        'Timeframe': tf,
                        'R:R Target': rr['label'],
                        'Stop Mode': mode.capitalize(),
                        'Signals': len(rets),
                        'Hit TP': tp_count,
                        'Hit SL': sl_count,
                        'Timeout': timeout_count,
                        'Win Rate (%)': round(win_rate, 2),
                        'Avg Net Ret (%)': round(rets.mean(), 4),
                        'Expectancy (%)': round(expectancy, 4),
                        'Net Profit Factor': round(profit_factor, 4),
                        'Portfolio Return (%)': round(cum_return, 2),
                        'Max Drawdown (%)': round(max_dd, 2),
                        'Sharpe (365d)': round(sharpe, 2),
                        'Sortino (365d)': round(sortino, 2)
                    })

df_results = pd.DataFrame(backtest_results)
print(f'Institutional Backtesting Complete! Evaluated {len(df_results)} strategy-timeframe-RR scenarios.')
df_results.sort_values('Net Profit Factor', ascending=False).head(10)

Institutional Backtesting Complete! Evaluated 162 strategy-timeframe-RR scenarios.


,Strategy,Timeframe,R:R Target,Stop Mode,Signals,Hit TP,Hit SL,Timeout,Win Rate (%),Avg Net Ret (%),Expectancy (%),Net Profit Factor,Portfolio Return (%),Max Drawdown (%),Sharpe (365d),Sortino (365d)
150,RSI Oversold Bounce,1h,1:3.0 R:R,Static,80,23,53,4,33.75,0.0456,0.0456,1.0655,2.62,-10.61,0.53,4.04
147,RSI Oversold Bounce,1h,1:2.0 R:R,Static,80,25,46,9,41.25,0.0290,0.0290,1.0351,0.96,-8.67,0.30,1.65
144,RSI Oversold Bounce,1h,1:1.5 R:R,Static,80,37,41,2,47.50,0.0212,0.0212,1.0287,0.68,-10.49,0.25,1.16
123,BB Squeeze Breakout,1h,1:3.0 R:R,Static,89,23,65,1,26.97,-0.1960,-0.1960,0.6993,-16.57,-23.42,-3.06,-15.34
90,"Supertrend Bearish (10,3)",1h,1:1.5 R:R,Static,76,30,46,0,39.47,-0.2610,-0.2610,0.6974,-12.74,-14.47,-3.26,-10.31
93,"Supertrend Bearish (10,3)",1h,1:2.0 R:R,Static,76,23,51,2,32.89,-0.3068,-0.3068,0.6780,-14.83,-15.97,-3.42,-12.70
141,VWMA Cross (8/21 Volume),1h,1:3.0 R:R,Static,169,38,122,9,27.81,-0.2879,-0.2879,0.6140,-39.41,-41.87,-4.28,-19.49
99,StochRSI Oversold,1h,1:1.5 R:R,Static,239,82,154,3,35.15,-0.3341,-0.3341,0.6074,-41.93,-42.06,-4.53,-18.68
111,StochRSI Overbought,1h,1:2.0 R:R,Static,274,74,185,15,31.02,-0.3827,-0.3827,0.6035,-51.11,-53.17,-4.53,-16.95
120,BB Squeeze Breakout,1h,1:2.0 R:R,Static,89,25,61,3,30.34,-0.3451,-0.3451,0.5850,-18.84,-21.95,-4.77,-18.22


## 5. Performance Comparison: 15m Timeframe Leaderboard

In [6]:
df_top = df_results[(df_results['Timeframe'] == '15m') & (df_results['Signals'] >= 10)].copy()
df_top = df_top.sort_values('Net Profit Factor', ascending=False).reset_index(drop=True)
df_top.index = df_top.index + 1
df_top.index.name = 'Rank'

print('='*110)
print('INSTITUTIONAL 15-MIN INTRADAY LEADERBOARD (NET OF 0.21% FRICTION & SLIPPAGE)')
print('='*110)
df_top[['Strategy', 'R:R Target', 'Stop Mode', 'Signals', 'Win Rate (%)', 
        'Avg Net Ret (%)', 'Net Profit Factor', 'Portfolio Return (%)', 'Max Drawdown (%)', 'Sharpe (365d)']].head(12)

INSTITUTIONAL 15-MIN INTRADAY LEADERBOARD (NET OF 0.21% FRICTION & SLIPPAGE)


,Strategy,R:R Target,Stop Mode,Signals,Win Rate (%),Avg Net Ret (%),Net Profit Factor,Portfolio Return (%),Max Drawdown (%),Sharpe (365d)
Rank,,,,,,,,,,
1,RSI Oversold Bounce,1:2.0 R:R,Static,78,37.18,-0.1683,0.5767,-8.45,-10.47,-4.96
2,RSI Oversold Bounce,1:1.5 R:R,Static,78,46.15,-0.1498,0.5685,-7.55,-8.52,-5.05
3,RSI Oversold Bounce,1:3.0 R:R,Static,78,29.49,-0.1744,0.5252,-12.83,-16.12,-5.89
4,StochRSI Oversold,1:2.0 R:R,Static,211,31.28,-0.2424,0.4301,-29.04,-29.04,-7.68
5,StochRSI Overbought,1:2.0 R:R,Static,241,29.46,-0.2431,0.4185,-32.48,-32.48,-8.04
6,StochRSI Oversold,1:1.5 R:R,Static,211,36.49,-0.2535,0.3673,-30.11,-30.11,-9.02
7,"Supertrend Bullish (10,3)",1:1.5 R:R,Static,79,34.18,-0.2496,0.3629,-12.37,-12.69,-9.09
8,StochRSI Overbought,1:1.5 R:R,Static,241,34.44,-0.2511,0.3580,-33.31,-33.41,-9.41
9,RSI Oversold Bounce,1:1.5 R:R,Breakeven,78,32.05,-0.2184,0.3564,-10.78,-10.78,-9.02


## 6. Visualizing Risk Management Impact

In [7]:
# Bar chart comparing Net Profit Factor across Stop Modes (Static vs Break-Even vs Trailing)
df_15m_rr2 = df_results[(df_results['Timeframe'] == '15m') & (df_results['R:R Target'] == '1:2.0 R:R')].copy()

fig = px.bar(
    df_15m_rr2,
    x='Strategy',
    y='Net Profit Factor',
    color='Stop Mode',
    barmode='group',
    title='15m Net Profit Factor Across Dynamic Stop Modes (1:2.0 R:R)',
    template='plotly_dark'
)
fig.add_hline(y=1.0, line_dash='dash', line_color='red', annotation_text='Breakeven Line (PF=1.0)')
fig.update_layout(height=500)
fig.update_xaxes(tickangle=30, tickfont_size=9)
fig.show()

In [8]:
# Scatter plot: Net Profit Factor vs Win Rate (bubble size = signal count)
fig2 = px.scatter(
    df_top.head(20),
    x='Win Rate (%)',
    y='Avg Net Ret (%)',
    size='Signals',
    color='Net Profit Factor',
    hover_name='Strategy',
    text='Strategy',
    title='Expectancy vs Win Rate (Bubble Size = Signal Count)',
    template='plotly_dark',
    color_continuous_scale='Viridis'
)
fig2.add_hline(y=0, line_dash='dash', line_color='gray')
fig2.update_traces(textposition='top center', textfont_size=9)
fig2.update_layout(height=550)
fig2.show()